# 📏 Phase 2.5: Ground Truth Thickness Extraction

Once your team has generated the annotation masks (and refined them if necessary), this notebook calculates the **maximum horizontal frost thickness (in mm)** for all 20 ROIs on every image.

This generates the `ground_truth_thickness.csv` file, which serves as the label targets for our neural network.

In [ ]:
from google.colab import drive
import os
import cv2
import numpy as np
import json
import csv
import glob

# Mount Google Drive
drive.mount('/content/drive')

# UPDATE THESE PATHS to match your Google Drive setup
MASKS_PATH = "/content/drive/MyDrive/Frost_Annotations"
ROI_JSON_PATH = "/content/drive/MyDrive/config/roi_coordinates.json" # You must upload the config folder to your drive
OUTPUT_CSV = "/content/drive/MyDrive/ground_truth_thickness.csv"

PIXELS_PER_MM = 3.4

In [ ]:
def extract_thickness(mask_roi):
    # Count white pixels horizontally across rows
    row_thicknesses = np.sum(mask_roi > 0, axis=1) / 255.0
    if len(row_thicknesses) == 0: return 0.0
    
    max_pixels = np.max(row_thicknesses)
    return round(max_pixels / PIXELS_PER_MM, 3)

if not os.path.exists(ROI_JSON_PATH):
    print(f"Error: {ROI_JSON_PATH} not found. Please upload it to Drive.")
else:
    with open(ROI_JSON_PATH, 'r') as f:
        rois = json.load(f)['rois']
        
    mask_files = glob.glob(os.path.join(MASKS_PATH, "**", "*_mask.png"), recursive=True)
    print(f"Found {len(mask_files)} mask files to process.")
    
    results = []
    for mask_path in mask_files:
        folder_name = os.path.basename(os.path.dirname(mask_path))
        img_name = os.path.basename(mask_path).replace("_mask.png", ".png")
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
            
        row_data = {"Folder": folder_name, "Image": img_name}
        
        for roi in rois:
            roi_id = roi['id']
            x1, y1, x2, y2 = roi['x1'], roi['y1'], roi['x2'], roi['y2']
            mask_crop = mask[y1:y2, x1:x2]
            row_data[f"ROI_{roi_id}"] = extract_thickness(mask_crop)
            
        results.append(row_data)
        
    if results:
        fieldnames = ["Folder", "Image"] + [f"ROI_{i}" for i in range(1, 21)]
        with open(OUTPUT_CSV, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for row in results:
                writer.writerow(row)
        print(f"✅ Successfully saved thickness data to {OUTPUT_CSV}")
